# TL;DR
Train a grayscale ResNet-18 from scratch on MNIST; source has no warmup number.

**Inputs:** MNIST, downloaded automatically into the selected work directory.

**Outputs:** Printed epoch losses and final test accuracy; no checkpoint in the source workflow.

Original Colab workflow; saved outputs cleared for publication. See the Python version for local execution.

# ResNet18 on MNIST

This notebook uses the same MNIST data pipeline as the earlier CNN: `ToTensor()` only, with no normalization, resizing, or augmentation.

In Google Colab, choose **Runtime > Change runtime type > T4 GPU** before running the cells.

In [ ]:
import torch
import torch.nn as nn
from torch import optim
from torch.utils.data import DataLoader
from torchvision import datasets, models
from torchvision.transforms import ToTensor

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

BATCH_SIZE = 100
NUM_EPOCHS = 10
LEARNING_RATE = 0.001
torch.manual_seed(42)

In [ ]:
# Same data conversion as the original CNN: no normalization or augmentation.
train_data = datasets.MNIST(root='data', train=True, transform=ToTensor(), download=True)
test_data = datasets.MNIST(root='data', train=False, transform=ToTensor(), download=True)

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(train_data.data.size(), test_data.data.size())

In [ ]:
# Standard ResNet18 residual stages, adapted only for grayscale 28x28 inputs.
model = models.resnet18(weights=None, num_classes=10)
model.conv1 = nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1, bias=False)
model.maxpool = nn.Identity()
model = model.to(device)

loss_function = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
print(model)

In [ ]:
for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        loss = loss_function(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f'Epoch {epoch + 1}/{NUM_EPOCHS}: loss = {total_loss / len(train_loader):.4f}')

In [ ]:
model.eval()
correct = total = 0
with torch.no_grad():
    for images, labels in test_loader:
        predictions = model(images.to(device)).argmax(dim=1).cpu()
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

accuracy = correct / total
print(f'ResNet18 test accuracy: {accuracy:.2%} ({correct}/{total})')